# MFDFA extraction and station-specific validation

This companion notebook documents the MFDFA implementation used for the archived feature table. The executable implementation is in `mfdfa.py`; the extraction driver is `extract_features.py`, and the nested validation is implemented in `evaluate_models.py`.

The supplied inputs contain 43,363 rows and 3,000 waveform samples per row plus a station identifier. The experiment selects the same 28,022 serial numbers at the nine adopted stations. No additional filtering, resampling, imputation, or row removal is performed.

## Setup and safety

The default setting below reads the archived outputs only. Set `RUN_FULL_EXPERIMENT = True` to invoke the extraction/validation driver. Because `data/mfdfa_features.csv` is supplied, the driver validates and reuses it. Raw inputs are required only to regenerate the table from waveforms.

In [ ]:
from pathlib import Path
import json
import pandas as pd

PACKAGE_DIR = Path.cwd()
if not (PACKAGE_DIR / 'mfdfa.py').exists():
    raise RuntimeError('Run this notebook from the release directory.')

RUN_FULL_EXPERIMENT = False

## Corrected mathematical implementation

The implementation uses non-overlapping segments constructed from both ends of the integrated profile, the logarithmic limit at q = 0, and the Legendre transformation `alpha(q) = d tau(q)/dq`, `f(alpha(q)) = q alpha(q) - tau(q)`. The reported generalized dimensions satisfy `D0 = -tau(0)`, `D1 = alpha(1)`, and `D2 = tau(2)`.

In [ ]:
import mfdfa

q_orders = list(range(-5, 6))
scales_for_supplied_waveforms = mfdfa.findscales(3000)
{
    'q_orders': q_orders,
    'number_of_scales': len(scales_for_supplied_waveforms),
    'minimum_scale_samples': int(scales_for_supplied_waveforms.min()),
    'maximum_scale_samples': int(scales_for_supplied_waveforms.max()),
}

## Optional full regeneration

In [ ]:
if RUN_FULL_EXPERIMENT:
    from extract_features import main
    main()
else:
    print('Full regeneration skipped. Set RUN_FULL_EXPERIMENT=True to run it.')

## Audited completed results

These values are stored with the out-of-fold predictions and are checked by `verify_release.py`. The headline accuracy is the unweighted arithmetic mean of the nine independently evaluated station accuracies, not the accuracy of a pooled global model.

In [ ]:
verification_path = PACKAGE_DIR / 'results' / 'aggregate_metrics.json'
verification = json.loads(verification_path.read_text())
verification

In [ ]:
station_metrics = pd.read_csv(PACKAGE_DIR / 'results' / 'station_oof_metrics.csv')
station_metrics

## Interpretation and limitations

The corrected experiment produced a station-macro accuracy of 94.28%, balanced accuracy of 94.28%, macro-precision of 93.56%, and macro-F1 of 93.87%. The original NPY array contains 3,000 samples per waveform and does not store the sampling frequency; therefore, it does not by itself establish the manuscript's 300 s at 50 Hz statement. A duplicate audit also found 27 exact within-station waveform pairs (54 rows, 0.193%); event identifiers are unavailable, so full event-wise uniqueness cannot be verified from these inputs alone.